# progress info

> Defines the data structure used to represent progress state

In [ ]:
#| default_exp progress_info

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass
import time
from typing import Optional

In [ ]:
#| export
@dataclass
class ProgressInfo:
    """Structured progress information"""
    progress: float                      # Percentage completion (0-100)
    current: Optional[int] = None        # Current iteration count
    total: Optional[int] = None          # Total iterations expected
    rate: Optional[str] = None           # Processing rate (e.g., "50.5 it/s")
    elapsed: Optional[str] = None        # Time elapsed since start
    remaining: Optional[str] = None      # Estimated time remaining
    description: Optional[str] = None    # Progress bar description/label
    raw_output: str = ""                 # Raw output string (if any)
    timestamp: float = None              # Unix timestamp when created
    bar_id: Optional[str] = None         # Unique identifier for this progress bar
    position: Optional[int] = None       # Display position for multi-bar scenarios
    
    def __post_init__(self):
        """Set timestamp to current time if not provided"""
        if self.timestamp is None:
            self.timestamp = time.time()
    
    def to_dict(
        self
    ) -> dict: # Dictionary with all progress fields for JSON serialization
        """Convert to dictionary for JSON serialization"""
        return {
            'progress': self.progress,
            'current': self.current,
            'total': self.total,
            'description': self.description,
            'rate': self.rate,
            'elapsed': self.elapsed,
            'remaining': self.remaining,
            'bar_id': self.bar_id,
            'position': self.position,
            'timestamp': self.timestamp,
            'raw_output': self.raw_output
        }

In [ ]:
#| export
from typing import Dict, Any, List

def serialize_job_snapshot(
    snapshot: Optional[Dict[str, Any]]  # Job snapshot dictionary from ProgressMonitor
) -> Optional[Dict[str, Any]]:  # JSON-serializable dictionary or None if input is None
    """Convert a job snapshot to JSON-serializable format"""
    if not snapshot:
        return None
    
    result = dict(snapshot)
    
    # Convert bars dict with ProgressInfo objects
    if 'bars' in result and result['bars']:
        result['bars'] = {
            k: v.to_dict() if hasattr(v, 'to_dict') else v 
            for k, v in result['bars'].items()
        }
    
    # Convert latest ProgressInfo
    if 'latest' in result and result['latest']:
        latest = result['latest']
        result['latest'] = latest.to_dict() if hasattr(latest, 'to_dict') else latest
    
    # Convert history if present
    if 'history' in result and result['history']:
        result['history'] = [
            h.to_dict() if hasattr(h, 'to_dict') else h 
            for h in result['history']
        ]
    
    return result

def serialize_all_jobs(
    jobs: Dict[str, Dict[str, Any]]  # Dictionary mapping job IDs to job snapshots
) -> Dict[str, Optional[Dict[str, Any]]]:  # Dictionary mapping job IDs to serialized snapshots
    """Convert all jobs from monitor.all() to JSON-serializable format"""
    return {
        job_id: serialize_job_snapshot(job) 
        for job_id, job in jobs.items()
    }

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()